# Linear Regression — Closed Form Solution and Mathematical Derivation

## 1. Problem Setup

Assume we have a dataset

$$
\{(x_i,y_i)\}_{i=1}^{N}
$$

where

$$
x_i \in \mathbb{R}^{D}
$$

is the feature vector and

$$
y_i \in \mathbb{R}
$$

is the target variable.

The linear regression model assumes

$$
y_i = w_0 + \sum_{j=1}^{D} w_j x_{ij} + \epsilon_i
$$

where the noise term follows

$$
\epsilon_i \sim N(0,\sigma^2)
$$

---

# 2. Matrix Representation

Define the design matrix

$$
X =
\begin{bmatrix}
x_{11} & x_{12} & \dots & x_{1D} \\
x_{21} & x_{22} & \dots & x_{2D} \\
\vdots & \vdots & \dots & \vdots \\
x_{N1} & x_{N2} & \dots & x_{ND}
\end{bmatrix}
$$

and the target vector

$$
y =
\begin{bmatrix}
y_1 \\
y_2 \\
\vdots \\
y_N
\end{bmatrix}
$$

The regression model becomes

$$
y = Xw + \epsilon
$$

---

# 3. Adding the Bias Term

To incorporate the intercept term we augment the design matrix with a column of ones

$$
X =
\begin{bmatrix}
1 & x_{11} & x_{12} & \dots & x_{1D} \\
1 & x_{21} & x_{22} & \dots & x_{2D} \\
\vdots & \vdots & \vdots & \dots & \vdots \\
1 & x_{N1} & x_{N2} & \dots & x_{ND}
\end{bmatrix}
$$

The weight vector becomes

$$
w =
\begin{bmatrix}
w_0 \\
w_1 \\
\vdots \\
w_D
\end{bmatrix}
$$

---

# 4. Objective Function

Linear regression minimizes the **sum of squared errors**

$$
L(w) =
\sum_{i=1}^{N}
\left(
y_i -
\sum_{j=0}^{D} x_{ij} w_j
\right)^2
$$

Using vector notation

$$
L(w) = ||y - Xw||_2^2
$$

---

# 5. Expanding the Loss Function

The squared norm expands as

$$
||y - Xw||_2^2 =
(y - Xw)^T (y - Xw)
$$

Expanding the expression

$$
= y^T y - 2y^T Xw + w^T X^T X w
$$

---

# 6. Minimizing the Loss Function

To find the optimal weights we compute the derivative with respect to $w$

$$
\frac{\partial L(w)}{\partial w}
$$

Taking the derivative

$$
\frac{\partial}{\partial w}
\left(
y^T y - 2y^T Xw + w^T X^T X w
\right)
$$

The derivatives of each term are

$$
\frac{\partial (y^Ty)}{\partial w} = 0
$$

$$
\frac{\partial (-2y^TXw)}{\partial w} = -2X^Ty
$$

$$
\frac{\partial (w^T X^T X w)}{\partial w} = 2X^TXw
$$

Therefore

$$
\nabla_w L(w) =
2X^TXw - 2X^Ty
$$

---

# 7. Normal Equation

To find the minimum we set the gradient to zero

$$
2X^TXw - 2X^Ty = 0
$$

Dividing by 2

$$
X^TXw = X^Ty
$$

This equation is called the **Normal Equation**.

---

# 8. Closed Form Solution

Solving for $w$

$$
w = (X^TX)^{-1}X^Ty
$$

This is the **closed form solution for linear regression**.

---

# 9. Numerical Stability and Pseudo-Inverse

In practice the matrix

$$
X^TX
$$

may not be invertible.

This happens when

- features are linearly dependent
- number of features exceeds number of observations

To handle this we use the **Moore–Penrose pseudo-inverse**

$$
(X^TX)^{+}
$$

The solution becomes

$$
w = (X^TX)^{+} X^T y
$$

This is exactly what is implemented in the code.

---

# 10. Implementation Formula

The final weight computation is

$$
w =
(X^TX)^{+} X^T y
$$

which corresponds to the implementation

```python
self.weights = np.linalg.pinv(X.T @ X) @ X.T @ y

In [1]:
class LinearRegression:
    def __init__(self,add_bias=True):
        self.add_bias = add_bias

        self.weights = None



    def fit(self,X,y):
        
        X = np.asarray(X)
        y = np.asarray(y)

        if X.ndim==1:
            X = X.reshape(-1,1)

        y = y.reshape(-1)
            



        
        N   = len(X)
        
        if self.add_bias:
            X = np.hstack((np.ones((N,1)),X))
            

        self.weights = np.linalg.pinv(X.T @ X) @ X.T @ y
        print(f"The final weights are : {self.weights}")

    def predict(self,X):
        X = np.asarray(X)

        if X.ndim==1:
            X = X.reshape(-1,1)



        
        N  = len(X)
        if self.add_bias:
            X = np.hstack((np.ones((N,1)),X))
            
        return X @ self.weights
        
            

## 1. Simple Linear Relationship (Ideal Case)

$$
y = 3x_1 - 2x_2 + \epsilon
$$

Closed-form solution for linear regression:

$$
\hat{w} = (X^TX)^{-1}X^Ty
$$

**Behaviour of Weights**

- Features are independent.
- The matrix $X^TX$ is well-conditioned.
- The inversion is numerically stable.
- Estimated weights converge close to the true coefficients.

$$
\hat{w} \approx [3, -2]
$$

In [2]:
import numpy as np

np.random.seed(0)

n = 5

X = np.random.randn(n, 2)

true_w = np.array([3, -2])

y = X @ true_w + 0.1 * np.random.randn(n)


In [3]:
model = LinearRegression(add_bias = False)
model.fit(X,y)
print(f"Predictions : {model.predict(X)} \n")

The final weights are : [ 3.03336296 -1.96143687]
Predictions : [ 4.56612794 -1.426503    7.5818501   3.17884062 -1.11846328] 



---

## 2. Multicollinearity (Highly Correlated Features)
Feature relationship:

$$
x_2 \approx x_1
$$

**Behaviour of Weights**

- Columns of $X$ become nearly linearly dependent.
- $X^TX$ becomes **ill-conditioned**.
- Small noise in the dataset causes **large fluctuations in the estimated weights**.
- Individual coefficients become unstable even though predictions remain reasonable.

Example:

$$
\hat{w} \approx [-73, 77]
$$

Even though the combined effect still approximates the true signal.

In [4]:
np.random.seed(0)

n = 5

x1 = np.random.randn(n)

x2 = x1 + 0.001 * np.random.randn(n)

X = np.column_stack((x1, x2))

true_w = np.array([2, 2])

y = X @ true_w + 0.1 * np.random.randn(n)

In [5]:
model = LinearRegression(add_bias = False)
model.fit(X,y)
print(f"Predictions : {model.predict(X)} \n")

The final weights are : [-73.26204794  77.28875712]
Predictions : [7.02779318 1.68474786 3.92939501 9.01544755 7.55184755] 



---

## 3. Near Zero Variance Features
Feature variance:

$$
Var(X_j) \approx 0
$$

**Behaviour of Weights**

- Features barely change across observations.
- The signal in $X$ is extremely weak.
- Noise dominates the regression.

Result:

- Weights become unstable.


In [6]:
np.random.seed(0)

n = 5

X = 0.0001 * np.random.randn(n, 2)

true_w = np.array([5, -3])

y = X @ true_w + 0.1 * np.random.randn(n)

In [7]:
print(f' The variance of X : {np.var(X)}')

 The variance of X : 9.352422005002838e-09


In [8]:
model = LinearRegression(add_bias = False)
model.fit(X,y)
print(f"Predictions : {model.predict(X)} \n")

The final weights are : [338.62957545 382.63126508]
Predictions : [0.0750473  0.11888654 0.02584733 0.0263814  0.01221549] 



---

## 4. Perfect Collinearity (Singular Matrix)
Feature relationship:

$$
x_2 = 2x_1
$$

**Behaviour of Weights**

The matrix becomes singular:

$$
\det(X^TX) = 0
$$

Therefore

$$
(X^TX)^{-1}
$$

does **not exist**.

Result:

- No unique solution exists.
- Infinite weight vectors can produce identical predictions.
- Methods such as **pseudo-inverse** which is used in this case is required.

In [9]:
np.random.seed(0)

n = 5

x1 = np.random.randn(n)

x2 = 2 * x1

X = np.column_stack((x1, x2))

true_w = np.array([1, 2])

y = X @ true_w 

In [10]:
print(f"The determinant of X.T @ X : {np.linalg.det(X.T @ X )}")

The determinant of X.T @ X : 0.0


In [11]:
model = LinearRegression(add_bias = False)
model.fit(X,y)
print(f"Predictions : {model.predict(X)} \n")

The final weights are : [1. 2.]
Predictions : [ 8.82026173  2.00078604  4.89368992 11.204466    9.33778995] 



---

## 5. Underdetermined System (More Features Than Samples)
Relationship:

$$
p > n
$$

where  

- $p$ = number of features  
- $n$ = number of samples  

**Behaviour of Weights**

- The rank of $X$ is at most $n$.
- The system has **infinitely many solutions**.

The regression equation

$$
Xw = y
$$

can be satisfied by many different weight vectors.

Solutions are typically obtained using:

- Minimum norm solution (pseudo-inverse) which is used in this case
- Regularization (Ridge regression)

In [12]:
np.random.seed(0)

n = 5
p = 50

X = np.random.randn(n, p)

true_w = np.random.randn(p)

y = X @ true_w + 0.1 * np.random.randn(n)

In [13]:
model = LinearRegression(add_bias = False)
model.fit(X,y)
print(f"Predictions : {model.predict(X)} \n")

The final weights are : [-0.10623999 -0.11136885 -0.43124286 -0.3919487  -0.47300028  0.45805836
 -0.14201176 -0.07429359  0.14296211  0.07477587  0.1046509  -0.06178374
 -0.27745826  0.0986795  -0.11846011  0.06993972 -0.30063095 -0.03592027
 -0.06301069  0.27370706  0.58913    -0.20583815  0.05292019  0.13572699
 -0.41453004  0.18523307 -0.1795158   0.17534444 -0.18615978 -0.16772571
 -0.19080392  0.03314869  0.08019731  0.06786881  0.16663961  0.11648544
 -0.12529594 -0.16226198 -0.0275877   0.05008068 -0.09932578  0.40894239
  0.31135702 -0.19379914  0.44903831  0.29467354  0.08339156  0.17310358
  0.23057061  0.12122122]
Predictions : [-10.17398213   3.19288587   5.10979935   2.63636519  -2.01101907] 



---

## 6. Overdetermined System (Typical Case)
Relationship:

$$
n >> p
$$

**Behaviour of Weights**

- There are more equations than unknowns.
- An exact solution may not exist.
- Linear regression finds the weights that minimize the squared error:

$$
\min_w ||y - Xw||^2
$$

This produces a **stable least-squares estimate** close to the true weights.

In [14]:
np.random.seed(0)

n = 500
p = 3

X = np.random.randn(n, p)

true_w = np.array([2, -1, 4])

y = X @ true_w + 0.2 * np.random.randn(n)

In [15]:
model = LinearRegression(add_bias = False)
model.fit(X,y)

The final weights are : [ 1.99882116 -1.01090097  3.9935952 ]


---

## 7. High Noise Dataset
Noise dominates the signal:

$$
Var(\epsilon) >> Var(Xw)
$$

**Behaviour of Weights**

- Model struggles to recover the true relationship.
- The coefficient estimates have **high variance**.


In [16]:
np.random.seed(0)

n = 10

X = np.random.randn(n, 2)

true_w = np.array([3, -2])
noise =  5 * np.random.randn(n)

y1 = X @ true_w 
y = y1 + noise

In [17]:
print(f"Noise Variance : {np.var(noise)}")
print(f"XW Variance : {np.var(y1)}")

Noise Variance : 49.2373423855057
XW Variance : 9.139630052942113


In [18]:
model = LinearRegression(add_bias = False)
model.fit(X,y)
print(f"Predictions : {model.predict(X)} \n")
print(f"True y : {y}")

The final weights are : [ 2.80795678 -3.64300414]
Predictions : [ 3.49560838 -5.41532924  8.80424948  3.21920215 -1.78564612 -4.89345628
  1.69369845  0.03077182  4.94270187  3.99055489] 

True y : [-8.27310646  1.72252053 11.87941072 -0.55784543 10.21791956 -9.74824467
  2.26855573 -0.27167821 12.55644982  9.99418843]


---

## 8. Non-Linear Relationship (Linear Model Misspecified)
True relationship:

$$
y = x^2
$$

But linear regression assumes:

$$
y = wx + b
$$

**Behaviour of Weights**

- The model cannot capture the curvature.
- Residuals show systematic patterns.
- The fitted line represents only a **linear approximation** of a nonlinear function.

In [19]:
np.random.seed(0)

n = 5

x = np.random.randn(n)

X = x.reshape(-1, 1)

y = x**2 + 0.1 * np.random.randn(n)

In [20]:
model = LinearRegression(add_bias = False)
model.fit(X,y)
print(f"Predictions : {model.predict(X)} \n")
print(f"True y : {y}")

The final weights are : [1.89664971]
Predictions : [3.34578938 0.75895805 1.85632312 4.25018944 3.54210333] 

True y : [3.01415289 0.25513463 0.94279232 5.01128045 3.5288327 ]


---

## 9. Outlier Dataset
Linear regression minimizes:

$$
\sum_{i=1}^{n}(y_i - \hat{y}_i)^2
$$

**Behaviour of Weights**

- Large residuals are squared.
- Outliers receive disproportionately high influence.
- The regression line shifts toward extreme observations.
- Estimated coefficients become biased.

Robust methods such as **Huber regression or RANSAC** handle this better.

In [21]:
np.random.seed(0)

n = 5

X = np.random.randn(n, 1)

y = 3 * X[:,0] + np.random.randn(n)

y[0] = 50
y[1] = -40

In [22]:
model = LinearRegression(add_bias = False)
model.fit(X,y)
print(f"Predictions : {model.predict(X)} \n")
print(f"True y : {y}")

The final weights are : [7.9270868]
Predictions : [13.98379606  3.17208092  7.75854095 17.7637549  14.80429429] 

True y : [ 50.         -40.           2.78485674   6.61946075   6.01327247]


---

## 10. Constant Feature Dataset
Feature relationship:

$$
x_2 = 2
$$

**Behaviour of Weights**

- If a bias term is added, the design matrix contains two identical columns.
- This produces **perfect multicollinearity**.

Consequences:

- $X^TX$ becomes singular.
- The weight vector is **not uniquely identifiable**.
- Multiple coefficient combinations produce the same predictions.

In [23]:
np.random.seed(0)

n = 5


x1 = np.ones(n)
x2 = 2*x1

X = np.column_stack((x1, x2))

y = 2 * x1 + np.random.randn(n)

In [24]:
print(f"The determinant of X.T @ X : {np.linalg.det(X.T @ X )}")
print(f"The variance of X : {np.var(X ,axis=0)}")

print(f"This model will always predict mean of training y data : {np.mean(y)} \n")


The determinant of X.T @ X : 0.0
The variance of X : [0. 0.]
This model will always predict mean of training y data : 3.4502797455584107 



In [25]:
model = LinearRegression(add_bias = False)
model.fit(X,y)

print(f"Predictions : {model.predict(X)} \n")
print(f"True y : {y}")

The final weights are : [0.69005595 1.3801119 ]
Predictions : [3.45027975 3.45027975 3.45027975 3.45027975 3.45027975] 

True y : [3.76405235 2.40015721 2.97873798 4.2408932  3.86755799]


---